# Track H: AI Foundations - Gate H1
## Calling an LLM API with Structured Output

- **Author**: Misiko
- **Gate**: Current Gate 22 of 39 (Track H1: AI Landscape, LLMs & Tooling)
- **Objective**: Execute an LLM API call using secure environment variables and enforce a strict, typed structured response schema (JSON / Pydantic).


### Step 1: Import Required Libraries
- os: Standard library to access system environment variables.
- dotenv.load_dotenv: Reads key-value pairs from .env and sets them in os.environ to avoid hardcoding secrets.
- pydantic: Data validation library used to declare strict typed schemas.
- google.genai: Modern official Google GenAI SDK to interact with Gemini models.


In [1]:
import os
import json
from dotenv import load_dotenv
from pydantic import BaseModel, Field
from google import genai

print("All libraries imported successfully.")


All libraries imported successfully.


### Step 2: Load API Credentials Securely
**Pass Criterion**: *API keys must be in environment variables and never hardcoded in source code or notebooks.*


In [2]:
# Load environment variables from the .env file
load_dotenv()

# Retrieve the API key from the environment
api_key = os.environ.get("GEMINI_API_KEY")

if not api_key:
    raise ValueError("GEMINI_API_KEY not found! Please define it in your .env file.")

# Verify key exists without exposing sensitive credentials
masked_key = f"{api_key[:4]}...{api_key[-4:]}" if len(api_key) > 8 else "***"
print(f"API Key loaded securely from environment: {masked_key}")

# Initialize the Gemini client
client = genai.Client(api_key=api_key)
print("Gemini client successfully initialized.")


API Key loaded securely from environment: AQ.A...-vNQ
Gemini client successfully initialized.


### Step 3: Define the Structured Output Schema
We use Pydantic to specify an exact data contract. The LLM is constrained to output JSON conforming strictly to this structure.


In [3]:
class AIConceptAnalysis(BaseModel):
    subfield: str = Field(description="The primary AI subfield (ML, DL, NLP, CV, or GenAI)")
    concept_name: str = Field(description="Name of the concept or technology")
    summary: str = Field(description="A concise technical summary of how it works")
    primary_tool: str = Field(description="The dominant framework or library used (e.g., PyTorch, Hugging Face)")
    real_world_example: str = Field(description="A named real-world production use case")
    key_takeaway: str = Field(description="One sentence summary of its core significance")

print("Structured schema 'AIConceptAnalysis' defined.")


Structured schema 'AIConceptAnalysis' defined.


### Step 4: Call LLM API with Structured Output Configuration
We instruct gemini-3.6-flash to analyze 'Self-Attention in Transformers', passing AIConceptAnalysis as the 
esponse_schema and setting 
esponse_mime_type to pplication/json.


In [4]:
prompt = (
    "Analyze the technical concept 'Self-Attention in Transformers' as part of the modern AI landscape. "
    "Provide a detailed, accurate evaluation matching the requested schema."
)

print("Dispatching request to Gemini API...")

response = client.models.generate_content(
    model="gemini-3.6-flash",
    contents=prompt,
    config={
        "response_mime_type": "application/json",
        "response_schema": AIConceptAnalysis,
    },
)

print("Response received from API.")


Direct use of automatic function calling (AFC) in Models.generate_content is not recommended. Instead, we recommend to use AFC in Chat.send_message. Similarly, direct use of AFC in Models.generate_content_stream is not recommended. Instead, we recommend to use AFC in Chat.send_message_stream.


Dispatching request to Gemini API...
Response received from API.


### Step 5: Validate and Print Structured Output
We parse the model's JSON response back through Pydantic to ensure full type-validation and then format it cleanly.


In [5]:
# Parse and validate the raw JSON text into our Pydantic model
structured_data = AIConceptAnalysis.model_validate_json(response.text)

# Pretty-print formatted JSON
print("=== Structured Response (Validated JSON) ===")
print(json.dumps(structured_data.model_dump(), indent=2))
print("============================================")

print(f"\nVerified Fields:")
print(f"- Subfield:           {structured_data.subfield}")
print(f"- Concept:            {structured_data.concept_name}")
print(f"- Dominant Tool:      {structured_data.primary_tool}")
print(f"- Real-World Example: {structured_data.real_world_example}")


=== Structured Response (Validated JSON) ===
{
  "subfield": "GenAI",
  "concept_name": "Self-Attention in Transformers",
  "summary": "Self-attention computes dynamic scalar weights for token relationships by projecting input embeddings into Query, Key, and Value matrices, using scaled dot-product operations to allow the model to aggregate contextual information across an entire sequence simultaneously.",
  "primary_tool": "PyTorch",
  "real_world_example": "OpenAI ChatGPT processing long conversational histories for context-aware response generation.",
  "key_takeaway": "Self-attention enables parallel processing of sequential data while capturing long-range contextual dependencies, forming the architectural foundation of modern foundation models."
}

Verified Fields:
- Subfield:           GenAI
- Concept:            Self-Attention in Transformers
- Dominant Tool:      PyTorch
- Real-World Example: OpenAI ChatGPT processing long conversational histories for context-aware response gen